In [1]:
!pip install streamlit plotly pyngrok -q
print("✓ Semua dependencies terinstall")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 95.5 MB/s eta 0:00:00
✓ Semua dependencies terinstall


In [23]:
import os

print("Cek file yang dibutuhkan:")
required = {
    '06_dashboard.py'                          : 'File dashboard',
    'risk_scores_with_explanation.csv'    : 'Risk scores + GPT explanation',
    'judolguard_.csv'             : 'Features dataset',
    'model_metrics.txt'                   : 'Model metrics',
    'eda_behavioral_shift.png'            : 'EDA chart',
    'xgb_judolguard.pkl'                : 'Trained model',
}
all_ok = True
for path, desc in required.items():
    ok = os.path.exists(path)
    if not ok: all_ok = False
    print(f"  {'✓' if ok else '✗ MISSING'} {path}  ({desc})")

if not all_ok:
    print("\n⚠️  Ada file yang missing. Jalankan notebook sebelumnya dulu.")
else:
    print("\n✓ Semua file siap. Lanjut ke Cell 3 untuk deploy.")

Cek file yang dibutuhkan:
  ✗ MISSING 06_dashboard.py  (File dashboard)
  ✓ risk_scores_with_explanation.csv  (Risk scores + GPT explanation)
  ✓ judolguard_.csv  (Features dataset)
  ✓ model_metrics.txt  (Model metrics)
  ✓ eda_behavioral_shift.png  (EDA chart)
  ✓ xgb_judolguard.pkl  (Trained model)

⚠️  Ada file yang missing. Jalankan notebook sebelumnya dulu.


In [18]:
import os
print("Working directory:", os.getcwd())
print("\nIsi folder data/:")
os.makedirs('data', exist_ok=True)
print(os.listdir('data'))

Working directory: /content

Isi folder data/:
['risk_scores_with_explanation.csv', 'judolguard.csv', 'risk_scores.csv', '.ipynb_checkpoints']


In [19]:
# Jalankan ini di session yang sama dengan deploy
from google.colab import files

# Upload judolguard_features.csv dulu
uploaded = files.upload()
# Pilih file judolguard_features.csv dari laptop kamu

import os
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)

# Pindahkan ke folder data/
import shutil
for fname in uploaded.keys():
    shutil.move(fname, f'data/{fname}')
    print(f"✓ {fname} dipindah ke data/")

KeyboardInterrupt: 

In [20]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import IsolationForest
import xgboost as xgb
from sklearn.model_selection import train_test_split

# Load data
df = pd.read_csv('data/judolguard.csv')

FEATURE_COLS = [
    'hour_of_day', 'is_night', 'night_ratio_7d', 'night_ratio_14d', 'temporal_shift',
    'amount_log', 'amount_vs_avg_7d', 'total_amount_7d',
    'tx_count_24h', 'tx_count_7d', 'burst_score',
    'unique_recv_7d', 'unique_recv_24h', 'qris_ratio_7d',
    'drain_cycle_flag', 'dormant_flag'
]

X = df[FEATURE_COLS].fillna(0)
y = df['is_at_risk']

# Isolation Forest
iso = IsolationForest(n_estimators=200, contamination=0.35, random_state=42)
iso.fit(X[y==0])
raw = iso.score_samples(X)
df['anomaly_score'] = 1 - ((raw - raw.min()) / (raw.max() - raw.min()))

X['anomaly_score'] = df['anomaly_score']

# XGBoost
spw = len(y[y==0]) / len(y[y==1])
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    scale_pos_weight=spw, random_state=42, n_jobs=-1,
    eval_metric='aucpr', use_label_encoder=False
)
xgb_model.fit(X, y, verbose=False)

# Risk scores per akun
df['risk_prob']  = xgb_model.predict_proba(X)[:, 1]
df['risk_score'] = (df['risk_prob'] * 100).round(1)

def get_level(s):
    if s <= 30: return 'Low'
    elif s <= 60: return 'Medium'
    elif s <= 80: return 'High'
    else: return 'Critical'

def get_rec(l):
    return {
        'Low'     : 'Monitor pasif — tidak ada tindakan segera',
        'Medium'  : 'Kirim notifikasi edukasi ke nasabah',
        'High'    : 'Batasi nominal transfer harian, minta konfirmasi',
        'Critical': 'Eskalasi ke tim compliance & flag ke OJK'
    }[l]

def get_trigger(row):
    triggers = {
        'Aktivitas malam tinggi' : row['avg_night_ratio'],
        'Frekuensi tinggi'       : row['avg_tx_24h'] / 20,
        'Banyak penerima'        : row['avg_unique_recv'] / 10,
        'Velocity burst'         : row['avg_burst_score'] / 5,
        'Pergeseran ke malam'    : max(row['avg_temporal_shift'], 0),
        'Penggunaan QRIS tinggi' : row['avg_qris_ratio'],
    }
    top = sorted(triggers.items(), key=lambda x: x[1], reverse=True)[:2]
    return ' & '.join([t[0] for t in top])

risk = df.groupby('account_id').agg(
    risk_score_max       = ('risk_score', 'max'),
    profile              = ('profile', 'first'),
    is_at_risk_true      = ('is_at_risk', 'first'),
    n_transactions       = ('step', 'count'),
    avg_night_ratio      = ('night_ratio_7d', 'mean'),
    avg_tx_24h           = ('tx_count_24h', 'mean'),
    avg_unique_recv      = ('unique_recv_7d', 'mean'),
    avg_burst_score      = ('burst_score', 'mean'),
    avg_temporal_shift   = ('temporal_shift', 'mean'),
    avg_qris_ratio       = ('qris_ratio_7d', 'mean'),
).reset_index()

risk['final_risk_score'] = risk['risk_score_max'].round(1)
risk['risk_level']       = risk['final_risk_score'].apply(get_level)
risk['recommendation']   = risk['risk_level'].apply(get_rec)
risk['top_triggers']     = risk.apply(get_trigger, axis=1)
risk['explanation']      = 'Penjelasan belum di-generate.'

risk.to_csv('data/risk_scores.csv', index=False)
risk.to_csv('data/risk_scores_with_explanation.csv', index=False)

import os; os.makedirs('models', exist_ok=True)
joblib.dump(xgb_model, 'models/xgb_judolguard.pkl')

print(f"✓ risk_scores.csv tersimpan: {len(risk)} akun")
print(f"✓ Distribusi:\n{risk['risk_level'].value_counts().to_string()}")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [10:15:14] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✓ risk_scores.csv tersimpan: 620 akun
✓ Distribusi:
risk_level
Critical    325
Medium      132
Low         122
High         41


In [24]:
import subprocess
import threading
import time
import os

# ── Opsi A: pakai pyngrok (gratis, tidak perlu akun) ──────
def deploy_ngrok():
    from pyngrok import ngrok, conf

    # Kalau punya authtoken ngrok (gratis di ngrok.com), uncomment baris ini:
    # conf.get_default().auth_token = "PASTE_NGROK_AUTHTOKEN_KAMU"

    # Kill proses streamlit yang mungkin masih jalan
    os.system("pkill -f streamlit 2>/dev/null")
    time.sleep(1)

    # Jalankan streamlit di background
    def run_streamlit():
        subprocess.run([
            "streamlit", "run", "06_dashboard.py",
            "--server.port", "8501",
            "--server.headless", "true",
            "--server.enableCORS", "false",
            "--server.enableXsrfProtection", "false"
        ])

    t = threading.Thread(target=run_streamlit, daemon=True)
    t.start()

    # Tunggu streamlit ready
    print("Menjalankan Streamlit...", end="")
    for _ in range(15):
        time.sleep(1)
        print(".", end="", flush=True)
    print()

    # Buat tunnel ngrok
    try:
        public_url = ngrok.connect(8501)
        print("\n" + "="*55)
        print("  ✓ DASHBOARD BERHASIL DIDEPLOY!")
        print("="*55)
        print(f"\n  🌐 URL: {public_url}")
        print(f"\n  Buka URL di atas di browser kamu")
        print(f"  Tunjukkan URL ini ke juri saat demo")
        print("\n  ⚠️  Jangan tutup cell ini selama demo berlangsung")
        print("="*55)
        return public_url
    except Exception as e:
        print(f"\n  Error ngrok: {e}")
        print("  Coba Opsi B di bawah")

# ── Opsi B: LocalTunnel (backup jika ngrok error) ─────────
def deploy_localtunnel():
    os.system("pkill -f streamlit 2>/dev/null")
    time.sleep(1)

    def run_streamlit():
        subprocess.run([
            "streamlit", "run", "06_dashboard.py",
            "--server.port", "8502",
            "--server.headless", "true",
        ])

    t = threading.Thread(target=run_streamlit, daemon=True)
    t.start()
    time.sleep(5)

    # Install localtunnel
    os.system("npm install -g localtunnel 2>/dev/null")

    # Jalankan localtunnel
    lt_process = subprocess.Popen(
        ["lt", "--port", "8502", "--subdomain", "judolguard"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    time.sleep(3)
    output = lt_process.stdout.readline().decode()
    print("\n" + "="*55)
    print("  ✓ DASHBOARD BERHASIL DIDEPLOY! (LocalTunnel)")
    print("="*55)
    print(f"\n  🌐 URL: {output.strip()}")
    print(f"  (Jika subdomain judolguard tidak tersedia,")
    print(f"   URL tetap muncul dengan subdomain random)")
    print("="*55)

In [25]:
from pyngrok import conf, ngrok
import subprocess, threading, time, os

# ← Ganti dengan authtoken dari dashboard.ngrok.com/get-started/your-authtoken
conf.get_default().auth_token = "3D4RffEbfxuNhdyDlTAYgoAUQzd_wtYvdDSBVDtHNPqJvpHB"

# Kill streamlit yang mungkin masih jalan
os.system("pkill -f streamlit 2>/dev/null")
time.sleep(1)

# Jalankan streamlit di background
def run_streamlit():
    subprocess.run([
        'streamlit', 'run', '06_dashboard.py',
        '--server.port', '8501',
        '--server.headless', 'true',
        '--server.enableCORS', 'false',
        '--server.enableXsrfProtection', 'false'
    ])

threading.Thread(target=run_streamlit, daemon=True).start()

# Tunggu streamlit siap
print("Menjalankan Streamlit", end="")
for _ in range(15):
    time.sleep(1)
    print(".", end="", flush=True)

# Buat tunnel
url = ngrok.connect(8501)
print(f"\n\n{'='*50}")
print(f"  ✓ DASHBOARD LIVE!")
print(f"  🌐 {url}")
print(f"{'='*50}")
print(f"\n  Jangan tutup cell ini selama demo.")

Menjalankan Streamlit

.

..............

  ✓ DASHBOARD LIVE!
  🌐 NgrokTunnel: "https://flashbulb-campsite-smartly.ngrok-free.dev" -> "http://localhost:8501"

  Jangan tutup cell ini selama demo.
